# Response Generator — Checkpoint Recovery & Testing

**Safe to re-run:** This notebook will NOT delete any checkpoints.

1. Find the best checkpoint and save as `final_adapter` (if not already saved)
2. Load the adapter and run tests (interactive + test data evaluation)

In [ ]:
!pip install -q transformers>=4.45.0 peft>=0.13.0 bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets huggingface_hub

In [ ]:
import shutil, os
from google.colab import drive

if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive')

drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in to HuggingFace via Colab Secrets')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', '')
    if hf_token:
        login(token=hf_token)
        print('Logged in via env var')
    else:
        print('WARNING: No HF_TOKEN found!')

In [ ]:
import os, json, glob, torch

MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/response_generator'
ADAPTER_DIR = f'{DRIVE_CKPT_DIR}/final_adapter'
TEST_FILE = '/content/drive/MyDrive/response_generator_test.jsonl'
NUM_EPOCHS = 2

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} | VRAM: {vram:.1f} GB')
else:
    raise RuntimeError('No GPU detected!')

# List existing checkpoints (read-only, never delete)
checkpoints = sorted(glob.glob(f'{DRIVE_CKPT_DIR}/checkpoint-*'))
print(f'Found {len(checkpoints)} checkpoint(s): {[os.path.basename(c) for c in checkpoints]}')
print(f'final_adapter exists: {os.path.exists(ADAPTER_DIR) and len(os.listdir(ADAPTER_DIR)) > 1}')

## Part 1: Save final_adapter from best checkpoint

This cell is **safe**: it only READS checkpoints, never deletes them.
If `final_adapter` already exists, it skips entirely.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if os.path.exists(ADAPTER_DIR) and any(f.endswith('.safetensors') for f in os.listdir(ADAPTER_DIR)):
    print(f'final_adapter already saved at {ADAPTER_DIR}, skipping.')

elif not checkpoints:
    raise FileNotFoundError(
        f'No checkpoints found in {DRIVE_CKPT_DIR}! '
        'Please check your Google Drive path.'
    )

else:
    # Find best checkpoint by lowest eval_loss
    best_ckpt = None
    best_loss = float('inf')
    best_step = None

    for ckpt in checkpoints:
        sf = os.path.join(ckpt, 'trainer_state.json')
        if os.path.exists(sf):
            with open(sf) as f:
                st = json.load(f)
            print(f'{os.path.basename(ckpt)}: epoch={st.get("epoch", "?")}, step={st.get("global_step", "?")}')
            for log in st.get('log_history', []):
                if 'eval_loss' in log and log['eval_loss'] < best_loss:
                    best_loss = log['eval_loss']
                    best_step = log.get('step', 0)

    # Match best step to checkpoint
    if best_step is not None:
        for ckpt in checkpoints:
            if f'checkpoint-{best_step}' in ckpt:
                best_ckpt = ckpt
                break

    if best_ckpt is None:
        best_ckpt = checkpoints[-1]
        print(f'\nFallback to last checkpoint: {best_ckpt}')
    else:
        print(f'\nBest checkpoint (eval_loss={best_loss:.4f}): {best_ckpt}')

    # Load base model + adapter from checkpoint
    print('\nLoading base model...')
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map='auto',
    )

    print('Loading adapter from checkpoint...')
    adapter_files = [f for f in os.listdir(best_ckpt) if 'adapter' in f.lower()]
    print(f'  Adapter files: {adapter_files}')
    peft_model = PeftModel.from_pretrained(base_model, best_ckpt)

    # Save as final_adapter
    print('\nSaving as final_adapter...')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token

    os.makedirs(ADAPTER_DIR, exist_ok=True)
    peft_model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)

    import subprocess
    result = subprocess.run(['du', '-sh', ADAPTER_DIR], capture_output=True, text=True)
    print(f'Adapter saved to: {ADAPTER_DIR}')
    print(f'Size: {result.stdout.strip()}')

    # Free VRAM for testing
    del base_model, peft_model
    torch.cuda.empty_cache()
    print('VRAM freed.')

## Part 2: Load final_adapter for testing

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

MODEL_ID = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER_DIR = '/content/drive/MyDrive/response_generator/final_adapter'

test_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, device_map='auto',
)
test_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
test_model.eval()
print('Model loaded from final_adapter for testing.')

In [ ]:
import re

SYSTEM_PROMPT = (
    'You are a compassionate medical assistant. A patient has been assigned '
    'an appointment. Write a warm, clear appointment confirmation and practical '
    'pre-visit instructions. Keep the tone professional but reassuring. '
    'Format your response as:\n'
    'Confirmation: <one sentence confirming the appointment>\n'
    'Instructions: <2-4 specific pre-visit instructions>'
)

def generate_response(input_text, temperature=0.7):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': input_text},
    ]
    tokenized = test_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True,
    ).to(test_model.device)
    input_len = tokenized['input_ids'].shape[-1]

    with torch.inference_mode():
        outputs = test_model.generate(
            **tokenized, max_new_tokens=512, temperature=temperature,
            do_sample=True, top_p=0.9, pad_token_id=test_tokenizer.eos_token_id,
        )
    generated = test_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return generated


def check_format(text):
    has_conf = bool(re.search(r'Confirmation:', text)) or 'confirmed' in text.lower()
    has_inst = bool(re.search(r'Instructions:', text)) or 'instruction' in text.lower()
    return has_conf, has_inst


print('Helper functions ready.')

## 2.1 Interactive Test — 6 Hand-Crafted Examples

Cover different departments and urgency levels.

In [ ]:
test_cases = [
    {
        'input': (
            'Patient symptoms: I have been experiencing severe chest pain and shortness of breath for the past 2 days\n'
            'Assigned department: Cardiology\n'
            'Doctor: Dr. Sarah Chen\n'
            'Appointment: Monday at 09:00\n'
            'Urgency: Emergency'
        ),
        'label': 'Emergency cardiology case',
    },
    {
        'input': (
            'Patient symptoms: I have had recurring headaches and occasional blurred vision for 3 weeks\n'
            'Assigned department: Neurology\n'
            'Doctor: Dr. James Wilson\n'
            'Appointment: Wednesday at 14:00\n'
            'Urgency: Urgent'
        ),
        'label': 'Urgent neurology case',
    },
    {
        'input': (
            'Patient symptoms: Mild skin rash on my arms that appeared last week, slightly itchy\n'
            'Assigned department: Dermatology\n'
            'Doctor: Dr. Emily Park\n'
            'Appointment: Friday at 11:00\n'
            'Urgency: Routine'
        ),
        'label': 'Routine dermatology case',
    },
    {
        'input': (
            'Patient symptoms: Persistent stomach pain and acid reflux after meals for the past month\n'
            'Assigned department: Gastroenterology\n'
            'Doctor: Dr. Michael Brown\n'
            'Appointment: Tuesday at 10:30\n'
            'Urgency: Routine'
        ),
        'label': 'Routine gastro case',
    },
    {
        'input': (
            'Patient symptoms: High fever of 39.5C, body aches, and cough for 4 days, recently traveled abroad\n'
            'Assigned department: Infectious Disease\n'
            'Doctor: Dr. Lisa Wang\n'
            'Appointment: Monday at 08:00\n'
            'Urgency: Urgent'
        ),
        'label': 'Urgent infectious disease case',
    },
    {
        'input': (
            'Patient symptoms: Knee pain and swelling after a fall, unable to bear weight\n'
            'Assigned department: Orthopedics\n'
            'Doctor: Dr. Robert Kim\n'
            'Appointment: Thursday at 15:00\n'
            'Urgency: Urgent'
        ),
        'label': 'Urgent orthopedics case',
    },
]

print('=' * 70)
print('INTERACTIVE TEST: 6 Hand-Crafted Examples')
print('=' * 70)

for i, tc in enumerate(test_cases):
    print(f'\n{chr(9472) * 70}')
    print(f'[Test {i+1}] {tc["label"]}')
    print(f'{chr(9472) * 70}')
    print(f'INPUT:\n{tc["input"]}\n')

    output = generate_response(tc['input'])
    hc, hi = check_format(output)

    print(f'OUTPUT:\n{output}\n')
    print(f'Format: Confirmation={"YES" if hc else "NO"} | Instructions={"YES" if hi else "NO"}')

## 2.2 Test Data Evaluation (200 samples)

Set `EVAL_LIMIT = None` for full 6362 samples (slow, ~2-3 hours).

In [ ]:
import json, random, sys
from collections import defaultdict

TEST_FILE = '/content/drive/MyDrive/response_generator_test.jsonl'
EVAL_LIMIT = 200   # Set to None for full 6362 samples

with open(TEST_FILE) as f:
    test_data = [json.loads(line) for line in f]

if EVAL_LIMIT:
    random.seed(42)
    test_data = random.sample(test_data, min(EVAL_LIMIT, len(test_data)))

print(f'Loaded {len(test_data)} test samples')

# Metrics
has_conf, has_inst, format_ok = 0, 0, 0
total_length = 0
dept_counts = defaultdict(lambda: {'total': 0, 'format_ok': 0})
urg_counts = defaultdict(lambda: {'total': 0, 'format_ok': 0})
sample_outputs = []
total = len(test_data)

for i, sample in enumerate(test_data):
    output = generate_response(sample['input'])
    total_length += len(output)

    hc, hi = check_format(output)
    if hc: has_conf += 1
    if hi: has_inst += 1
    if hc and hi: format_ok += 1

    dept = sample.get('department', 'Unknown')
    urg = sample.get('urgency', 'Unknown')
    dept_counts[dept]['total'] += 1
    urg_counts[urg]['total'] += 1
    if hc and hi:
        dept_counts[dept]['format_ok'] += 1
        urg_counts[urg]['format_ok'] += 1

    if len(sample_outputs) < 5:
        sample_outputs.append({
            'input': sample['input'][:150],
            'generated': output[:300],
            'expected': sample['output'][:300],
        })

    done = i + 1
    pct = done / total
    bar = chr(9608) * int(pct * 30) + chr(9617) * (30 - int(pct * 30))
    comp = format_ok / done
    print(f'\r  {bar} {done}/{total} ({pct:.0%}) | compliance: {comp:.1%}', end='')
    sys.stdout.flush()

print(f'\n\nEvaluation complete.')

In [ ]:
n = len(test_data)

print('=' * 60)
print('RESPONSE GENERATOR TEST REPORT')
print('=' * 60)
print(f'Total samples:         {n}')
print(f'Format compliance:     {format_ok/n:.1%}')
print(f'Confirmation present:  {has_conf/n:.1%}')
print(f'Instructions present:  {has_inst/n:.1%}')
print(f'Avg response length:   {total_length/n:.0f} chars')

print(f'\n--- Per-Department ---')
print(f'{"Department":<22} {"Compliance":>10} {"Samples":>8}')
for dept, c in sorted(dept_counts.items()):
    comp = c['format_ok'] / c['total'] if c['total'] > 0 else 0.0
    print(f'{dept:<22} {comp:>10.1%} {c["total"]:>8}')

print(f'\n--- Per-Urgency ---')
print(f'{"Urgency":<22} {"Compliance":>10} {"Samples":>8}')
for urg, c in sorted(urg_counts.items()):
    comp = c['format_ok'] / c['total'] if c['total'] > 0 else 0.0
    print(f'{urg:<22} {comp:>10.1%} {c["total"]:>8}')

In [ ]:
print('=' * 70)
print('SAMPLE OUTPUTS: Generated vs Expected')
print('=' * 70)

for i, s in enumerate(sample_outputs):
    print(f'\n[Sample {i+1}]')
    print(f'INPUT:     {s["input"]}...')
    print(f'GENERATED: {s["generated"]}...')
    print(f'EXPECTED:  {s["expected"]}...')
    print(chr(9472) * 70)

In [ ]:
# Save results to Drive
results = {
    'total': n,
    'format_compliance': format_ok / n,
    'confirmation_rate': has_conf / n,
    'instructions_rate': has_inst / n,
    'avg_response_length': total_length / n,
    'department_report': {
        d: {'total': c['total'], 'format_compliance': c['format_ok'] / c['total'] if c['total'] > 0 else 0.0}
        for d, c in sorted(dept_counts.items())
    },
    'urgency_report': {
        u: {'total': c['total'], 'format_compliance': c['format_ok'] / c['total'] if c['total'] > 0 else 0.0}
        for u, c in sorted(urg_counts.items())
    },
}

output_path = '/content/drive/MyDrive/test_response_generator_results.json'
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {output_path}')